# Setup

In [8]:
!pip install -q "bibtexparser<2" requests rapidfuzz pandas

import bibtexparser, requests, re, time
import pandas as pd
from rapidfuzz import fuzz
from google.colab import files

HEADERS = {"User-Agent": "BOBR-Prototype/0.1"}


# #2 Import: .bib-Dateien einlesen



In [9]:
uploaded = files.upload()                       # .bib-Datei über GUI auswählen
filename = next(iter(uploaded))

try:
    entries = bibtexparser.loads(uploaded[filename].decode("utf-8")).entries
    if not entries:
        raise ValueError("Keine Einträge gefunden.")
    print(f"{len(entries)} Einträge aus '{filename}' geladen")
except Exception as err:                         # fehlerhafte Datei
    entries = []
    print(f"Datei konnte nicht gelesen werden: {err}")

Saving references.bib to references (6).bib
61 Einträge aus 'references (6).bib' geladen


# #7 Korrektheitsprüfung: Feldformate validieren (DOI, ISBN, URL)



In [14]:
DOI_RE = re.compile(r"^10\.\d{4,9}/[-._;()/:A-Za-z0-9]+$") # Muster Doi
URL_RE = re.compile(r"^https?://\S+$") # Muster URL

def valid_isbn13(s):
    d = [int(c) for c in s if c.isdigit()]
    if len(d) != 13: return False
    return sum(x * (1 if i % 2 == 0 else 3) for i, x in enumerate(d)) % 10 == 0

def valid_isbn10(s):
    if not re.match(r"^\d{9}[\dXx]$", s): return False
    total = sum((10 - i) * int(c) for i, c in enumerate(s[:9]))
    total += 10 if s[9] in "Xx" else int(s[9])
    return total % 11 == 0

def check_formats(e):
    probleme = []
    if (doi := e.get("doi")) and not DOI_RE.match(doi.strip()):
        probleme.append("DOI-Format ungültig")
    if (isbn := e.get("isbn")):
        clean = re.sub(r"[-\s]", "", isbn)
        if not (valid_isbn13(clean) or valid_isbn10(clean)):
            probleme.append("ISBN Prüfsumme falsch")
    if (url := e.get("url")) and not URL_RE.match(url.strip()):
        probleme.append("URL Format ungültig")
    return probleme


# #22  Abgleich Crossref + Korrektur vorschlag

In [15]:
def check_crossref(e):
    doi = e.get("doi")
    if not doi:
        return "kein DOI", "", ""
    try:
        r = requests.get(f"https://api.crossref.org/works/{doi.strip()}",
                         headers=HEADERS, timeout=10)
    except requests.RequestException:
        return "Netzwerkfehler", "", ""
    if r.status_code != 200:
        return "DOI nicht auflösbar", "", ""

    msg = r.json()["message"]
    cr_year  = msg.get("issued", {}).get("date-parts", [[None]])[0][0]
    cr_title = (msg.get("title") or [""])[0]

    # Jahr: Abweichung + konkreter Korrekturvorschlag
    if (y := e.get("year")) and cr_year and str(y) != str(cr_year):
        return "Jahr weicht ab", f"{y} ≠ {cr_year}", f"year = {cr_year}"

    # Titel: Abweichung + Vorschlag
    if e.get("title") and cr_title and fuzz.token_sort_ratio(e["title"], cr_title) < 85:
        return "Titel weicht ab", "", f"title = {cr_title}"

    return "ok", "", ""


Prüfen auf Vollständigkeit

In [19]:
REQUIRED = {
    "article":       {"author", "title", "journal", "year"},
    "book":          {"author", "title", "publisher", "year"},
    "inproceedings": {"author", "title", "booktitle", "year"},
}

def missing_fields(e):
    needed = REQUIRED.get(e.get("ENTRYTYPE", "").lower(), {"author", "title", "year"})
    return needed - set(e.keys())


Prüfen auf Konsistenz (doppelte einträge nur anders geschrieben bzw leicht verändert)

In [20]:
def find_duplicate_map(entries, threshold=90): # woher der treshhold
    """Ordnet jeder Eintrags-ID die IDs ähnlicher (möglicherweise doppelter) Einträge zu."""
    dup = {e.get("ID"): [] for e in entries}
    for i in range(len(entries)):
        for j in range(i + 1, len(entries)):
            t1, t2 = entries[i].get("title", ""), entries[j].get("title", "")
            if t1 and t2 and fuzz.token_sort_ratio(t1, t2) >= threshold:
                id1, id2 = entries[i].get("ID"), entries[j].get("ID")
                dup[id1].append(id2)
                dup[id2].append(id1)
    return dup


# Ergebnis

In [22]:
dup_map = find_duplicate_map(entries)          # einmal vorab berechnen

rows = []
for e in entries:
    cr_status, cr_detail, vorschlag = check_crossref(e)
    partner = dup_map.get(e.get("ID"), [])
    rows.append({
        "ID":                 e.get("ID"),
        "Titel":              (e.get("title") or "")[:40],
        "Vollständigkeit":    ", ".join(missing_fields(e)) or "ok",
        "Format (#7)":        ", ".join(check_formats(e)) or "ok",
        "Crossref (#22)":     cr_status,
        "Detail":             cr_detail,
        "Korrekturvorschlag": vorschlag,
        "Doppelte?":          ", ".join(partner) if partner else "–",
    })
    time.sleep(0.2)                            # API Limit

df = pd.DataFrame(rows)
df


,ID,Titel,Vollständigkeit,Format (#7),Crossref (#22),Detail,Korrekturvorschlag,Doppelte?
0,akerlof1970,The Market for ``Lemons'': Quality Uncer,ok,ok,kein DOI,,,–
1,hevner2004,Design Science in Information Systems Re,ok,ok,kein DOI,,,–
2,hevner2010,Design Research in Information Systems:,ok,ok,kein DOI,,,–
3,chapman2000,CRISP-DM 1.0: Step-by-step data mining g,ok,ok,kein DOI,,,–
4,vaswani2017,Attention Is All You Need,ok,ok,kein DOI,,,–
...,...,...,...,...,...,...,...,...
56,xiong2024confidence,Can {LLMs} Express Their Uncertainty? An,ok,ok,kein DOI,,,–
57,cizek2007,Standard Setting: A Guide to Establishin,ok,ok,Titel weicht ab,,title = Standard Setting,–
58,paukerkassirer1980,The Threshold Approach to Clinical Decis,ok,ok,ok,,,–
59,imaa2023,Hindsight or Foresight: Reassessing the,ok,ok,kein DOI,,,–


JSON speichern Ergebnis, vorbereitung für DB folgt

In [17]:
df.to_json("bobr_ergebnis.json", orient="records", force_ascii=False, indent=2)
files.download("bobr_ergebnis.json")            # lädt die Datei herunter
print("Ergebnis gespeichert als bobr_ergebnis.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Ergebnis gespeichert als bobr_ergebnis.json
